

# : Meeting Notes Auto-Summarizer

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

Writing up meeting notes by hand takes forever. Zoom AI Companion auto-generates summaries directly from the call transcript. Here we fine-tune BART (a model built for summarization) on dialogue-style text, then add a simple rule-based step that pulls out action items ('will do X', 'need to Y') from the generated summary.

### SECTION 1: Install & Import Libraries

In [1]:
!pip install -q transformers datasets accelerate evaluate rouge_score openai

import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
Enter your Groq API key: ··········


### SECTION 2: Load Meeting-Style Transcript Dataset (with robust fallback)

In [2]:
# heads up: the original "samsum" repo on HuggingFace used the old loading
# script style, and it was pulling data from an outside file server that
# isn't around anymore, plus newer `datasets` versions block scripts
# anyway. so we try a couple of working re-uploads, and if none of those
# work either, we just use a few made-up meeting chats.
def robust_load_dataset(repo_candidates, split="train"):
    for repo_id in repo_candidates:
        try:
            print(f"[dataset] Trying load_dataset('{repo_id}', split='{split}')...")
            ds = load_dataset(repo_id, split=split)
            print(f"[dataset] Loaded '{repo_id}' successfully ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] '{repo_id}' standard load failed: {type(e).__name__}: {e}")
        try:
            parquet_path = f"hf://datasets/{repo_id}@refs/convert/parquet/default/{split}/*.parquet"
            print(f"[dataset] Trying Parquet fallback: {parquet_path}")
            ds = load_dataset("parquet", data_files=parquet_path, split="train")
            print(f"[dataset] Parquet fallback for '{repo_id}' succeeded ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] Parquet fallback for '{repo_id}' failed: {type(e).__name__}: {e}")
    return None

train_data = robust_load_dataset(["knkarthick/samsum", "samsum", "Samsung/samsum"], split="train")
val_data = robust_load_dataset(["knkarthick/samsum", "samsum", "Samsung/samsum"], split="validation")

if train_data is None or val_data is None:
    # ---- Synthetic offline fallback -- guarantees the notebook always runs ----
    SYNTHETIC_MEETINGS = [
        {"dialogue": "Amy: Can we push the launch to next Friday?\nRaj: Why, is something blocking us?\nAmy: QA found two critical bugs in checkout.\nRaj: Okay, let's push it. I'll tell the stakeholders.\nAmy: Thanks, I'll get the fixes prioritized today.",
         "summary": "Amy and Raj agree to delay the launch to next Friday due to critical checkout bugs found by QA."},
        {"dialogue": "Sam: Did we finalize the Q3 budget?\nLena: Almost, marketing wants another 15% for ads.\nSam: That's a lot. Can we approve half for now?\nLena: I'll propose that in tomorrow's meeting.\nSam: Sounds good, send me the deck tonight.",
         "summary": "Sam and Lena discuss the Q3 budget; Lena will propose approving half of marketing's requested ad increase and send the deck tonight."},
        {"dialogue": "Priya: The client wants a demo before Thursday.\nJohn: We're not ready, the API integration isn't done.\nPriya: Can we do a partial demo with mocked data?\nJohn: Yeah, I can have that ready by Wednesday.\nPriya: Perfect, I'll schedule it.",
         "summary": "Priya and John agree to run a partial demo with mocked data by Wednesday since the API integration isn't finished."},
        {"dialogue": "Marco: Server costs jumped 40% this month.\nDina: Probably the new logging pipeline.\nMarco: Can we reduce retention to 7 days?\nDina: I'll check compliance requirements first.\nMarco: Okay, let me know by Friday.",
         "summary": "Marco and Dina investigate a 40% server cost increase from a new logging pipeline; Dina will check if reducing log retention to 7 days is compliant."},
        {"dialogue": "Tara: Are we still hiring for the backend role?\nOwen: Yes, we have two finalists.\nTara: When are the final interviews?\nOwen: This Thursday and Friday.\nTara: Great, send me their resumes before then.",
         "summary": "Tara and Owen confirm the backend hiring is proceeding with final interviews Thursday and Friday; Owen will send resumes beforehand."},
    ]
    expanded = SYNTHETIC_MEETINGS * 60  # replicate so the Trainer has a workable number of steps
    full_ds = Dataset.from_list(expanded)
    split_ds = full_ds.train_test_split(test_size=0.1, seed=42)
    train_data = split_ds["train"]
    val_data = split_ds["test"]
    print("[dataset] Using synthetic fallback: swap in a real Kaggle/HF dataset before reporting final metrics.")

print("Train dialogues:", len(train_data))
print(train_data[0])

[dataset] Trying load_dataset('knkarthick/samsum', split='train')...


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

[dataset] Loaded 'knkarthick/samsum' successfully (14731 rows).
[dataset] Trying load_dataset('knkarthick/samsum', split='validation')...
[dataset] Loaded 'knkarthick/samsum' successfully (818 rows).
Train dialogues: 14731
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


### SECTION 3: Tokenize

In [3]:
model_checkpoint = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_fn(batch):
    inputs = tokenizer(batch["dialogue"], max_length=768, truncation=True, padding="max_length")
    labels = tokenizer(batch["summary"], max_length=128, truncation=True, padding="max_length")
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_train = train_data.map(tokenize_fn, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(tokenize_fn, batched=True, remove_columns=val_data.column_names)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

### SECTION 4: Load Pretrained BART-large

In [4]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

### SECTION 5: Fine-Tune

In [5]:
use_fp16 = torch.cuda.is_available()  # fp16 only works with a real GPU -- Runtime > Change runtime type > T4 GPU
print("GPU available:", use_fp16)

training_args = Seq2SeqTrainingArguments(
    output_dir="./meeting-summarizer-checkpoints",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,     # BART-large is bigger -- smaller batch fits free T4 VRAM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # simulate an effective batch size of 16
    gradient_checkpointing=True,       # uses less memory, trains a little slower
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,
    predict_with_generate=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized_train, eval_dataset=tokenized_val,
    data_collator=data_collator, processing_class=tokenizer  # newer transformers renamed "tokenizer" to "processing_class"
)
trainer.train()

GPU available: True


Epoch,Training Loss,Validation Loss
1,2.395073,0.297296
2,0.929136,0.293642
3,0.791804,0.303476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2763, training_loss=1.1821027956619843, metrics={'train_runtime': 5705.4552, 'train_samples_per_second': 7.746, 'train_steps_per_second': 0.484, 'total_flos': 7.182814026085171e+16, 'train_loss': 1.1821027956619843, 'epoch': 3.0})

### SECTION 6: Evaluate with ROUGE

In [6]:
import evaluate
rouge = evaluate.load("rouge")

def generate_summary(dialogue):
    inputs = tokenizer(dialogue, return_tensors="pt", truncation=True, max_length=768)
    with torch.no_grad():
        output = model.generate(**{k: v.to(model.device) for k, v in inputs.items()}, max_new_tokens=100, num_beams=4)
    return tokenizer.decode(output[0], skip_special_tokens=True)

sample_val = val_data.select(range(100))
predictions = [generate_summary(d) for d in sample_val["dialogue"]]
references = sample_val["summary"]
results = rouge.compute(predictions=predictions, references=references)
print("ROUGE scores:", results)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

ROUGE scores: {'rouge1': np.float64(0.3915592101768282), 'rouge2': np.float64(0.19603254252968944), 'rougeL': np.float64(0.3015388000390422), 'rougeLsum': np.float64(0.30186288113486787)}


### SECTION 7: Extract Action Items (simple rule-based add-on)

In [7]:
import re

def extract_action_items(summary_text):
    action_verbs = r"\b(will|should|need to|must|has to|plans? to)\b"
    sentences = re.split(r"(?<=[.!?])\s+", summary_text)
    return [s for s in sentences if re.search(action_verbs, s, re.IGNORECASE)]

sample_summary = predictions[0]
print("Sample summary:", sample_summary)
print("Extracted action items:", extract_action_items(sample_summary))

Sample summary: A wants to get a puppy for her son. She took him to the animal shelter last Monday and he really liked the one she showed him. He wanted to take it home right away and named it Lemmy after his dead hamster - Lemmy - a Motorhead fan.
Extracted action items: []


### SECTION 8: Save Model

In [8]:
model.save_pretrained("./meeting-summarizer-model")
tokenizer.save_pretrained("./meeting-summarizer-model")
print("Model saved to ./meeting-summarizer-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./meeting-summarizer-model
